# 03 — Multi-Model Benchmarking, Cost-Loss Analysis & Threshold Tuning
**Project:** Insurance Claim Fraud Detection — Major Project Phase 1  
**Objective:** Rigorously benchmark multiple classifiers using 5-Fold Stratified Cross-Validation. Optimize decision threshold based on business financial loss ($50,000 FN vs $500 FP).

In [ ]:
from pathlib import Path

import sys

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



sys.path.insert(0, str(Path.cwd().parent))

from src.train import train_and_benchmark

from src.evaluate import find_optimal_threshold, compute_financial_loss

import joblib

## 1. Execute Benchmark & Pipeline Training Engine

In [ ]:
champion_pipeline, metadata = train_and_benchmark()

## 2. Cost-Benefit Threshold Curve
Plot the relationship between the decision threshold and business financial loss.

In [ ]:
# Load test split for threshold visualization

from src.data_loader import load_raw_data, parse_dates_and_target, get_stratified_split

clean_df = parse_dates_and_target(load_raw_data())

_, X_test, _, y_test = get_stratified_split(clean_df, test_size=0.20)

y_test_prob = champion_pipeline.predict_proba(X_test)[:, 1]



best_th, min_loss, df_cost = find_optimal_threshold(y_test.values, y_test_prob)



fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:red'

ax1.set_xlabel('Probability Decision Threshold (θ)')

ax1.set_ylabel('Total Expected Financial Loss ($)', color=color)

ax1.plot(df_cost['threshold'], df_cost['total_financial_loss'], color=color, lw=2.5)

ax1.axvline(best_th, color='black', linestyle='--', label=f'Optimal θ* = {best_th:.2f}')

ax1.axvline(0.50, color='gray', linestyle=':', label='Default θ = 0.50')

ax1.tick_params(axis='y', labelcolor=color)

ax1.legend(loc='upper right')



ax2 = ax1.twinx()

color = 'tab:blue'

ax2.set_ylabel('F1 Score / Recall', color=color)

ax2.plot(df_cost['threshold'], df_cost['f1_score'], color='blue', label='F1-Score', linestyle='-.')

ax2.plot(df_cost['threshold'], df_cost['recall'], color='green', label='Recall', linestyle='-')

ax2.tick_params(axis='y', labelcolor=color)

plt.title('Cost-Sensitive Decision Threshold Optimization')

plt.tight_layout()

plt.show()